In [12]:
import pandas as pd
import numpy as np

dataset = pd.read_csv("CKD.csv")   # make sure CKD.csv is in same folder as ipynb

print("Total Rows:", dataset.shape[0])
print("Total Columns:", dataset.shape[1])
print("\nColumns:\n", list(dataset.columns))
print("\nTarget distribution:\n", dataset["classification"].value_counts(dropna=False))

dataset.head()


Total Rows: 399
Total Columns: 25

Columns:
 ['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'classification']

Target distribution:
 classification
yes    249
no     150
Name: count, dtype: int64


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,2.0,76.459948,c,3.0,0.0,normal,abnormal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,yes,no,yes
1,3.0,76.459948,c,2.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,34.000000,12300.000000,4.705597,no,no,no,yes,poor,no,yes
2,4.0,76.459948,a,1.0,0.0,normal,normal,notpresent,notpresent,99.000000,...,34.000000,8408.191126,4.705597,no,no,no,yes,poor,no,yes
3,5.0,76.459948,d,1.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,poor,yes,yes
4,5.0,50.000000,c,0.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,36.000000,12400.000000,4.705597,no,no,no,yes,poor,no,yes


In [13]:
# Replace common missing marker
dataset = dataset.replace("?", np.nan)

# Clean target
dataset["classification"] = dataset["classification"].astype(str).str.strip().str.lower()
dataset["classification"] = dataset["classification"].map({"yes": 1, "no": 0})

# Separate X and y
X = dataset.drop("classification", axis=1)
y = dataset["classification"]

# Fill missing values:
# - For object columns: fill with mode
# - For numeric columns: fill with median
for col in X.columns:
    if X[col].dtype == "object":
        X[col] = X[col].astype(str).str.strip().str.lower().replace("nan", np.nan)
        X[col] = X[col].fillna(X[col].mode(dropna=True)[0])
    else:
        X[col] = pd.to_numeric(X[col], errors="coerce")
        X[col] = X[col].fillna(X[col].median())

# Convert categorical to dummy variables (nominal -> numeric)
X = pd.get_dummies(X, drop_first=True)

print("After encoding, X shape:", X.shape)


After encoding, X shape: (399, 27)


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=1/3, random_state=0, stratify=y
)

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

print("Train:", X_train.shape, "Test:", X_test.shape)


Train: (266, 27) Test: (133, 27)


In [16]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

models = {
    "LogisticRegression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=0, class_weight="balanced"),
    "KNN": KNeighborsClassifier(),
    "SVM(Default)": SVC()
}

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

research_rows = []

for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    research_rows.append({
        "Model": name,
        "CV Accuracy": scores["test_accuracy"].mean(),
        "CV Precision": scores["test_precision"].mean(),
        "CV Recall": scores["test_recall"].mean(),
        "CV F1": scores["test_f1"].mean(),
        "CV ROC-AUC": scores["test_roc_auc"].mean()
    })

research_df = pd.DataFrame(research_rows).sort_values(by="CV ROC-AUC", ascending=False)
research_df


,Model,CV Accuracy,CV Precision,CV Recall,CV F1,CV ROC-AUC
0,LogisticRegression,0.992453,1.000000,0.987879,0.993846,1.000000
1,RandomForest,0.981132,0.977143,0.993939,0.985158,1.000000
3,SVM(Default),0.988749,1.000000,0.981996,0.990861,1.000000
2,KNN,0.969881,1.000000,0.951693,0.975087,0.993485


In [17]:
from sklearn.model_selection import GridSearchCV

# NOTE: probability=True makes SVM slower.
# We use decision_function for ROC-AUC later (faster).
svc = SVC()

param_grid = {
    "kernel": ["linear", "rbf", "poly", "sigmoid"],
    "gamma": ["scale", "auto"],
    "C": [0.1, 1, 10, 100, 1000]
}

grid = GridSearchCV(
    estimator=svc,
    param_grid=param_grid,
    scoring="f1_weighted",
    cv=5,
    n_jobs=-1,
    verbose=2,
    refit=True
)

grid.fit(X_train, y_train)

print("\nBest Parameters:", grid.best_params_)
print("Best CV Score (F1_weighted):", grid.best_score_)


Fitting 5 folds for each of 40 candidates, totalling 200 fits

Best Parameters: {'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}
Best CV Score (F1_weighted): 0.9887580070259545


In [18]:
results = pd.DataFrame(grid.cv_results_)

grid_table = results[[
    "mean_test_score",
    "std_test_score",
    "param_kernel",
    "param_gamma",
    "param_C",
    "rank_test_score"
]].sort_values("rank_test_score")

grid_table.head(15)   # screenshot this table


,mean_test_score,std_test_score,param_kernel,param_gamma,param_C,rank_test_score
13,0.988758,0.014978,rbf,auto,1.0,1
15,0.988758,0.014978,sigmoid,auto,1.0,1
11,0.988758,0.014978,sigmoid,scale,1.0,1
9,0.988758,0.014978,rbf,scale,1.0,1
21,0.988758,0.014978,rbf,auto,10.0,1
17,0.988758,0.014978,rbf,scale,10.0,1
18,0.988725,0.009206,poly,scale,10.0,7
22,0.988725,0.009206,poly,auto,10.0,7
14,0.988641,0.015224,poly,auto,1.0,9
30,0.985048,0.021764,poly,auto,100.0,10


In [19]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix, classification_report

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

# For ROC-AUC without probability=True
y_score = best_model.decision_function(X_test)

print("\n=== Final Test Metrics ===")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_score))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))



=== Final Test Metrics ===
Accuracy : 0.9774436090225563
Precision: 0.9878048780487805
Recall   : 0.9759036144578314
F1-score : 0.9818181818181818
ROC-AUC  : 0.9990361445783132

Confusion Matrix:
 [[49  1]
 [ 2 81]]

Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.98      0.97        50
           1       0.99      0.98      0.98        83

    accuracy                           0.98       133
   macro avg       0.97      0.98      0.98       133
weighted avg       0.98      0.98      0.98       133

